# Univariate Analysis

The objective of this notebook is to analyze each variable separately to understand its distribution, central tendencies (mean, median), and dispersion (variance, standard deviation). It also includes visualization of variables.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
SILVER_DIR = DATA_DIR / 'silver'
DATA_FILE = SILVER_DIR / 'df_fraud.parquet'

base_directory = 'notebooks/images/images_univar_analysis'
os.makedirs(base_directory, exist_ok=True)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import pie_plot, bar_plot, bool_analysis, numerical_analysis, categorical_analysis, detect_outliers
import numpy as np
import os
from IPython.display import Image
from scipy.stats import probplot, kstest


In [ ]:
df = pd.read_parquet(DATA_FILE, engine='fastparquet')
df

In [ ]:
df.info(show_counts=True)

Observations:

- No nulls

Observations:


- amount: there is an enormous variation of amount of transferred money and the standard deviation is very high (603.858). 50% of all transactions move below 74.872, 75% of all transactions move below 208.721 and the max amount is 92.445.200 so we will find very few but huge transactions (long tail).

- old_balance_orig, new_balance_orig, old_balance_dest, new_balance_dest: both have similar standard deviations and maximum values. It calls my attention that min, 25% and even 50% can be 0. That makes me wonder if those accounts have been opened or closed just to commit fraud. Some significant amount accounts have also completely completely emptied. 

- is_flagged_fraud: the mean is even lower but we have to take into consideration that this flag is activated only for transactions greater than 200.000. 

- name_orig and name_dest: some names are repeated. We will check if that has something to do with fraudulent behaviours. 

## Target (is_fraud) 

In [ ]:
bool_analysis(df["is_fraud"])

In [ ]:
pie_plot(df["is_fraud"])

Observations: 

Fraudulent transactions represent only 0.13% (1 in 773 transactions), this is a severe class imbalance.

What to do: 

- Accuracy cannot be the primary metric because a naive model would classify everything as no_fraud and would get a 99.9% of accuracy. Time to check precision, recall, f-1 score, AUC-ROC, precision-recall curves

- I need to resampling or weighting like SMORE, ADASYN or class weights in model training. Also useful to check ensemble methods

- Important to use cross-validation if possible 

- Adjust thresholds

# Categorical Values

## Type

In [ ]:
categorical_analysis(df["type"])

In [ ]:
pie_plot(df["type"])

In [ ]:
bar_plot(df["type"])

Observations:

- Dominant types are CASH_OUT & PAYMENT.
- DEBIT is rare (0.7%)

What to do:

- DEBIT is very rare but it still has enough rows to be considered 
- Check fraud rate per transaction type

## name_orig & name_dest

From the data quality notebook:

- The cardinality is very high (99.85%)
- There are no nulls or invalid values

The frequencies are 

In [ ]:
col = df['name_orig']

print("\nAbsolute Frequency:")
print(col.value_counts())

print("\nRelative Frequency (%):")
print(col.value_counts(normalize=True).mul(100).round(2).astype(str) + "%")



In [ ]:
col = df['name_dest']

print("\nAbsolute Frequency:")
print(col.value_counts())

print("\nRelative Frequency (%):")
print(col.value_counts(normalize=True).mul(100).round(2).astype(str) + "%")



In [ ]:

filename = 'barplot_nameorig_namedest.png'
plot_filename = os.path.join(base_directory, filename)
    
if not os.path.isfile(plot_filename):
    categorical_cols = [ 'name_orig', 'name_dest']

    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(20, 10))

    for ax, col in zip(axs, categorical_cols):
        top_categories = df[col].value_counts().nlargest(150)
        
        top_categories.plot(kind='bar', ax=ax)
        
        ax.set_title(f'Top 150 most frequent categories in {col}')
        ax.set_xlabel(col)
        ax.set_ylabel('Count')
        ax.grid(True)

    plt.tight_layout()
    plt.savefig(plot_filename)  
    plt.close(fig)  

Image(filename=plot_filename)

Observations:
- name_orig: some accounts have sent money 2-3 times (low repeat senders)
- name_dest: High-frequency receivers exist (> 100 transactions/account).

What to do:

- Engineer feats: reduce both variables to Client and Merchant using the initial letter of each name
- Analyse fraud rate vs. account activity level

In [ ]:
df['name_orig_type'] = (
    df['name_orig'].str[0].replace({'C': 'Client', 'M': 'Merchant'}))


In [ ]:
pie_plot(df['name_orig_type'])

In [ ]:
df['name_dest_type'] = (
    df['name_dest'].str[0].replace({'C': 'Client', 'M': 'Merchant'}))


In [ ]:
pie_plot(df['name_dest_type'])

Observations:

- name_orig is an ID, and all types of transactions have origin of "Client"
- name_dest is also an ID and with two types of names representing "Client" and "Merchant". Both well represented

What to do: 

- I don't think that the length of the code after the initial letter has any meaning but it would be a good idea to analyse it just in case. 
- Check the relationship between C-C and fraud and also C-M-fraud in the multivar analysis

# Numerical variables

### Step

In [ ]:
numerical_analysis(df['step'])

In [ ]:
column = df['step'] 

fig, axes = plt.subplots(3, 1, figsize=(18, 15))

# 1. Histogram
axes[0].hist(column, bins=100, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].set_title(f'Histogram: {column.name} Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel(f'{column.name}', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].axvline(column.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {column.mean():.1f}')
axes[0].axvline(column.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {column.median():.1f}')
axes[0].legend()
axes[0].grid(True, alpha=0.5)

# 2. Box plot
bp = axes[1].boxplot(column, vert=False, patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
axes[1].set_title(f'Box Plot: {column.name} Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel(f'{column.name}', fontsize=12)
axes[1].set_yticks([])  # No y-ticks needed
axes[1].grid(True, alpha=0.5, axis='x')

# 3. Transactions over time
transactions_per_step = column.value_counts().sort_index()
axes[2].plot(transactions_per_step.index, transactions_per_step.values, color='navy', linewidth=1.5)
axes[2].set_title(f'Transaction Volume Over {column.name}', fontsize=14, fontweight='bold')
axes[2].set_xlabel(f'{column.name}', fontsize=12)
axes[2].set_ylabel('Number of Transactions', fontsize=12)
axes[2].grid(True, alpha=0.5)

plt.tight_layout()
plt.show()


Observations: 

- The histogram shows a roughly uniform distribution with a slight right skew (skewness=0.375)
- No major gaps or missing time periods visible
- Mean (243.4) is slightly higher than median (239.0), confirming the rightskew
- Data spans 743 unique time steps (1 to 743), suggesting ~31 days if steps are hours
- Some steps have higher transaction volumes than others, this could reflect daily/weekly patterns
- Box plot shows extreme outliers in time values indicating few transactions spread across the later time range. These outliers are expected and should be kept, as they reflect real activity patterns

Next steps:
1. In bivariate analysis: Check if certain time periods correlate with different patterns
2. Consider creating cyclical features (e.g., time_of_day, day_of_week) if step represents hours
3. May bin into time windows for pattern analysis
4. Investigate the volume fluctuations to understand if they follow a pattern


In [ ]:
df['step']

In [ ]:
#subset with the step column for univar analysis 
step_temp = df.copy()
step_temp = step_temp[['step']]

In [ ]:

step_temp['day'] = ((step_temp['step'] - 1) // 24) + 1
step_temp['hour'] = ((step_temp['step'] - 1) % 24)

def time_of_day(hour):
    if 6 <= hour < 12: return 'morning'
    elif 12 <= hour < 18: return 'afternoon'
    elif 18 <= hour < 24: return 'evening'
    else: return 'night'

step_temp['time_of_day'] = step_temp['hour'].apply(time_of_day)
step_temp['is_weekend'] = step_temp['day'].apply(lambda d: 'week_day' if d % 7 in (6, 0) else 'weekend')


In [ ]:
step_temp

In [ ]:
numerical_analysis(step_temp['day'])

In [ ]:
column = step_temp['day'] 

fig, axes = plt.subplots(3, 1, figsize=(18, 15))

# 1. Histogram
axes[0].hist(column, bins=100, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].set_title(f'Histogram: {column.name} Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel(f'{column.name}', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].axvline(column.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {column.mean():.1f}')
axes[0].axvline(column.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {column.median():.1f}')
axes[0].legend()
axes[0].grid(True, alpha=0.5)

# 2. Box plot
bp = axes[1].boxplot(column, vert=False, patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
axes[1].set_title(f'Box Plot: {column.name} Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel(f'{column.name}', fontsize=12)
axes[1].set_yticks([])  # No y-ticks needed
axes[1].grid(True, alpha=0.5, axis='x')

# 3. Transactions over time
transactions_per_step = column.value_counts().sort_index()
axes[2].plot(transactions_per_step.index, transactions_per_step.values, color='navy', linewidth=1.5)
axes[2].set_title(f'Transaction Volume Over {column.name}', fontsize=14, fontweight='bold')
axes[2].set_xlabel(f'{column.name}', fontsize=12)
axes[2].set_ylabel('Number of Transactions', fontsize=12)
axes[2].grid(True, alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
column = step_temp['hour']

fig, axes = plt.subplots(3, 1, figsize=(18, 15))

# 1. Histogram
axes[0].hist(column, bins=100, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].set_title(f'Histogram: {column.name} Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel(f'{column.name}', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].axvline(column.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {column.mean():.1f}')
axes[0].axvline(column.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {column.median():.1f}')
axes[0].legend()
axes[0].grid(True, alpha=0.5)

# 2. Box plot
bp = axes[1].boxplot(column, vert=False, patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
axes[1].set_title(f'Box Plot: {column.name} Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel(f'{column.name}', fontsize=12)
axes[1].set_yticks([])  # No y-ticks needed
axes[1].grid(True, alpha=0.5, axis='x')

# 3. Transactions over time
transactions_per_step = column.value_counts().sort_index()
axes[2].plot(transactions_per_step.index, transactions_per_step.values, color='navy', linewidth=1.5)
axes[2].set_title(f'Transaction Volume Over {column.name}', fontsize=14, fontweight='bold')
axes[2].set_xlabel(f'{column.name}', fontsize=12)
axes[2].set_ylabel('Number of Transactions', fontsize=12)
axes[2].grid(True, alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
column = step_temp['time_of_day']

fig, axes = plt.subplots(3, 1, figsize=(18, 15))

# 1. Histogram
axes[0].hist(column, bins=100, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].set_title(f'Histogram: {column.name} Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel(f'{column.name}', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].axvline(column.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {column.mean():.1f}')
axes[0].axvline(column.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {column.median():.1f}')
axes[0].legend()
axes[0].grid(True, alpha=0.5)

# 2. Box plot
bp = axes[1].boxplot(column, vert=False, patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
axes[1].set_title(f'Box Plot: {column.name} Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel(f'{column.name}', fontsize=12)
axes[1].set_yticks([])  # No y-ticks needed
axes[1].grid(True, alpha=0.5, axis='x')

# 3. Transactions over time
transactions_per_step = column.value_counts().sort_index()
axes[2].plot(transactions_per_step.index, transactions_per_step.values, color='navy', linewidth=1.5)
axes[2].set_title(f'Transaction Volume Over {column.name}', fontsize=14, fontweight='bold')
axes[2].set_xlabel(f'{column.name}', fontsize=12)
axes[2].set_ylabel('Number of Transactions', fontsize=12)
axes[2].grid(True, alpha=0.5)

plt.tight_layout()
plt.show()


import numpy as np
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
